# detector — VRAI RUN (dataset entier)
Detection (Binoculars) + attribution par perplexite sur les 250 textes et les 5 modeles.

**Runtime -> GPU.** Les gros modeles sont charges en 4-bit pour tenir sur un T4.

In [1]:
# 1) Dependances
!pip -q install transformers accelerate bitsandbytes scikit-learn pandas numpy
import torch; print('GPU dispo :', torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 12.8 MB/s eta 0:00:00
GPU dispo : True


## 2) Login HuggingFace
**Obligatoire** : gemma-4 et Mistral sont gated (accepte leurs licences sur HF avant).

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## 3) Charge dataset_clean.csv

In [3]:
from google.colab import files
up = files.upload()   # -> selectionne dataset_clean.csv

Saving dataset_clean.csv to dataset_clean.csv


## 4) Le detecteur (Binoculars + attribution, 5 modeles, 4-bit)

In [4]:
"""
detector.py
------------------------------------------------------------------
Detecteur du projet "Can an AI Recognize Another AI?"

Deux parties, a partir de dataset.csv :

  PARTIE A - DETECTION (humain vs IA) avec Binoculars, implemente CORRECTEMENT :
     * observateur et performeur PARTAGENT le meme tokenizer
     * vraie CROSS-PERPLEXITE = cross-entropie entre les distributions
       completes des deux modeles (pas juste la log-prob du vrai token !)
     * score = log_ppl(observateur) / cross_ppl(observateur, performeur)
       -> texte IA = score plus BAS

  PARTIE B - ATTRIBUTION (quel modele) par perplexite :
     * chaque modele candidat (open-weight) calcule la perplexite du texte
     * on attribue au modele le moins "surpris" (perplexite la plus basse)
     * normalisation z-score par modele pour une comparaison equitable

Dependances :
  pip install torch transformers pandas scikit-learn numpy
------------------------------------------------------------------
"""

import gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             confusion_matrix)

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_CSV = "dataset_clean.csv"      # le CSV nettoye (surface artefacts retires)
OUT_CSV     = "dataset_scored.csv"

# --- Binoculars : une PAIRE qui partage le meme tokenizer ---
# Leger (recommande pour commencer) :
OBSERVER  = "Qwen/Qwen2.5-0.5B"
PERFORMER = "Qwen/Qwen2.5-0.5B-Instruct"
# Version du papier (plus lourde) : "tiiuae/falcon-7b" / "tiiuae/falcon-7b-instruct"

# --- Attribution : les modeles candidats (open-weight) ---
# cle = nom court tel qu'il apparait dans la colonne model_name du CSV
# valeur = id HuggingFace a charger
CANDIDATE_MODELS = {
    "Qwen3-4B-Instruct-2507":     "Qwen/Qwen3-4B-Instruct-2507",
    "gemma-4-E4B-it":             "google/gemma-4-E4B-it",
    "SmolLM3-3B":                 "HuggingFaceTB/SmolLM3-3B",
    "Mistral-7B-Instruct-v0.3":   "mistralai/Mistral-7B-Instruct-v0.3",
    "Phi-4-mini-instruct":        "microsoft/Phi-4-mini-instruct",
}

MAX_LEN = 1024
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.float16 if DEVICE == "cuda" else torch.float32

# Chargement en 4-bit pour que les gros modeles (7B) tiennent sur un GPU Colab (T4).
# Evite le "device_map=auto" qui decharge sur le disque et provoque l'erreur meta/cuda.
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)


# ============================================================
# OUTILS PERPLEXITE
# ============================================================

def _load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if DEVICE == "cuda":
        # 4-bit, tout sur le GPU 0 (pas d'offload disque)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=BNB, device_map={"": 0}).eval()
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE).eval()
    return tok, model

def _free(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def token_ids(text, tok):
    return tok(text, return_tensors="pt", truncation=True,
               max_length=MAX_LEN).input_ids.to(DEVICE)

def log_perplexity(logits, target_ids):
    """Moyenne des NLL des vrais tokens = log-perplexite."""
    logprobs = torch.log_softmax(logits.float(), dim=-1)
    idx = torch.arange(target_ids.shape[0], device=logits.device)
    nll = -logprobs[idx, target_ids]
    return nll.mean().item()

def cross_perplexity(obs_logits, perf_logits, chunk=64):
    """
    VRAIE cross-perplexite : moyenne, sur les positions, de la cross-entropie
    entre la distribution complete de l'observateur (p) et celle du
    performeur (q) : H(p, q) = - sum_v p(v) * log q(v).
    (Chunke sur les positions pour ne pas saturer la memoire.)
    """
    T = obs_logits.shape[0]
    total = 0.0
    for s in range(0, T, chunk):
        p = torch.softmax(obs_logits[s:s+chunk].float(), dim=-1)     # observateur
        logq = torch.log_softmax(perf_logits[s:s+chunk].float(), -1) # performeur
        ce = -(p * logq).sum(dim=-1)      # [chunk]
        total += ce.sum().item()
    return total / T


# ============================================================
# PARTIE A - DETECTION (BINOCULARS)
# ============================================================

def compute_binoculars_scores(df):
    print(f"\n=== Binoculars : {OBSERVER}  vs  {PERFORMER} ===")
    tok = AutoTokenizer.from_pretrained(OBSERVER)
    # petits modeles (0.5B) : fp16 directement sur le GPU, sans offload
    obs = AutoModelForCausalLM.from_pretrained(
        OBSERVER, torch_dtype=DTYPE).to(DEVICE).eval()
    perf = AutoModelForCausalLM.from_pretrained(
        PERFORMER, torch_dtype=DTYPE).to(DEVICE).eval()

    # verifie que la paire partage bien le meme vocabulaire
    assert obs.config.vocab_size == perf.config.vocab_size, \
        "Observateur et performeur DOIVENT partager le meme tokenizer !"

    scores = []
    for i, text in enumerate(df["text"].tolist()):
        ids = token_ids(text, tok)
        with torch.no_grad():
            o = obs(ids).logits[0]      # [L, V]
            p = perf(ids).logits[0]     # [L, V]
        # on predit le token t+1 depuis la position t
        o, p = o[:-1], p[:-1]
        targets = ids[0, 1:]
        log_ppl = log_perplexity(o, targets)     # observateur
        x_ppl   = cross_perplexity(o, p)          # observateur vs performeur
        scores.append(log_ppl / x_ppl)
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(df)}")

    _free(obs); _free(perf)
    df["binoculars_score"] = scores
    return df

def evaluate_detection(df):
    print("\n--- Evaluation DETECTION (humain vs IA) ---")
    # etiquette : 1 = IA (machine), 0 = humain
    y = (df["source_type"] == "model").astype(int).values
    b = df["binoculars_score"].values
    # IA -> score BAS, donc la "machine-ness" = -score
    auroc = roc_auc_score(y, -b)
    print(f"AUROC (detection IA) : {auroc:.3f}")

    # seuil calibre sur le train, evalue sur le test
    tr, te = df["split"] == "train", df["split"] == "test"
    cand = np.quantile(b[tr], np.linspace(0.05, 0.95, 50))
    best_tau, best_acc = cand[0], -1
    for tau in cand:                       # IA si score < tau
        acc = accuracy_score(y[tr], (b[tr] < tau).astype(int))
        if acc > best_acc:
            best_acc, best_tau = acc, tau
    pred_te = (b[te] < best_tau).astype(int)
    print(f"Seuil choisi : {best_tau:.4f}")
    print(f"Accuracy test : {accuracy_score(y[te], pred_te):.3f}")
    return auroc, best_tau


# ============================================================
# PARTIE B - ATTRIBUTION PAR PERPLEXITE
# ============================================================

def model_perplexities(df):
    """
    Retourne une matrice [n_textes, n_modeles] de log-perplexites :
    perplexite de chaque texte sous chaque modele candidat.
    """
    names = list(CANDIDATE_MODELS.keys())
    ppl = np.full((len(df), len(names)), np.nan)

    for j, name in enumerate(names):
        print(f"\n=== Perplexite sous {name} ===")
        tok, model = _load(CANDIDATE_MODELS[name])
        for i, text in enumerate(df["text"].tolist()):
            ids = token_ids(text, tok)
            with torch.no_grad():
                logits = model(ids).logits[0][:-1]
            ppl[i, j] = log_perplexity(logits, ids[0, 1:])
            if (i + 1) % 25 == 0:
                print(f"  {i+1}/{len(df)}")
        _free(model)
    return names, ppl

def evaluate_attribution(df, names, ppl):
    print("\n--- Evaluation ATTRIBUTION (quel modele) ---")
    # on n'evalue que sur les textes ECRITS par un des modeles candidats
    mask = df["source_type"].eq("model") & df["model_name"].isin(names)
    sub = df[mask].reset_index(drop=True)
    P = ppl[mask.values]

    # normalisation z-score PAR modele : enleve le biais "certains modeles
    # ont une perplexite globalement plus basse", pour comparer la surprise RELATIVE
    Pz = (P - np.nanmean(P, axis=0)) / (np.nanstd(P, axis=0) + 1e-8)

    def eval_pred(matrix, label):
        pred_idx = np.argmin(matrix, axis=1)             # modele le moins surpris
        pred = [names[k] for k in pred_idx]
        true = sub["model_name"].tolist()
        acc = accuracy_score(true, pred)
        f1  = f1_score(true, pred, average="macro")
        print(f"[{label}] accuracy={acc:.3f}  macro-F1={f1:.3f}  (hasard={1/len(names):.3f})")
        print("Matrice de confusion (lignes=vrai, cols=predit), ordre =", names)
        print(confusion_matrix(true, pred, labels=names))

    eval_pred(P,  "perplexite brute")
    eval_pred(Pz, "perplexite z-normalisee")


# ============================================================
# MAIN
# ============================================================

def main():
    df = pd.read_csv(DATASET_CSV)
    df = df[df["text"].astype(str).str.split().apply(len) >= 20].reset_index(drop=True)

    # PARTIE A
    df = compute_binoculars_scores(df)
    df.to_csv(OUT_CSV, index=False)
    evaluate_detection(df)

    # PARTIE B
    names, ppl = model_perplexities(df)
    np.save("perplexity_matrix.npy", ppl)
    evaluate_attribution(df, names, ppl)

    print(f"\nOK -> scores sauvegardes dans {OUT_CSV}")


## 5) Lance le run complet
Produit `dataset_scored.csv` + `perplexity_matrix.npy`, puis les telecharge.

In [5]:
main()

from google.colab import files
files.download('dataset_scored.csv')
files.download('perplexity_matrix.npy')


=== Binoculars : Qwen/Qwen2.5-0.5B  vs  Qwen/Qwen2.5-0.5B-Instruct ===


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

  25/250
  50/250
  75/250
  100/250
  125/250
  150/250
  175/250
  200/250
  225/250
  250/250

--- Evaluation ATTRIBUTION (quel modele) ---
[perplexite brute] accuracy=0.712  macro-F1=0.664  (hasard=0.200)
Matrice de confusion (lignes=vrai, cols=predit), ordre = ['Qwen3-4B-Instruct-2507', 'gemma-4-E4B-it', 'SmolLM3-3B', 'Mistral-7B-Instruct-v0.3', 'Phi-4-mini-instruct']
[[23  0  0  2  0]
 [ 0  0  0 25  0]
 [ 0  0 25  0  0]
 [ 0  0  0 25  0]
 [ 0  0  0  9 16]]
[perplexite z-normalisee] accuracy=0.776  macro-F1=0.781  (hasard=0.200)
Matrice de confusion (lignes=vrai, cols=predit), ordre = ['Qwen3-4B-Instruct-2507', 'gemma-4-E4B-it', 'SmolLM3-3B', 'Mistral-7B-Instruct-v0.3', 'Phi-4-mini-instruct']
[[24  1  0  0  0]
 [ 4 13  0  8  0]
 [ 0  8 17  0  0]
 [ 0  2  0 23  0]
 [ 0  5  0  0 20]]

OK -> scores sauvegardes dans dataset_scored.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>